In [ ]:
# ============================================================
# 01_csv_to_jsonl.py
# 목적:
#   파인튜닝 관리용 CSV를 학습용 JSONL로 변환
#
# 입력:
#   army_finetune_dataset_v3_100_revised.csv
#
# 출력:
#   train.jsonl
#   validation.jsonl
#   test.jsonl
# ============================================================

!pip -q install pandas

import os
import json
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")


# ------------------------------------------------------------
# 1. 경로 설정
# ------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/Army_Finetune")

CSV_PATH = BASE_DIR / "01_dataset" / "army_finetune_dataset_v3_100_revised.csv"
JSONL_DIR = BASE_DIR / "02_jsonl"
JSONL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = JSONL_DIR / "train.jsonl"
VAL_JSONL = JSONL_DIR / "validation.jsonl"
TEST_JSONL = JSONL_DIR / "test.jsonl"


# ------------------------------------------------------------
# 2. CSV 로드
# ------------------------------------------------------------
df = pd.read_csv(CSV_PATH)

print("원본 행 수:", len(df))
print("컬럼 목록:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. 필수 컬럼 확인
# ------------------------------------------------------------
required_cols = [
    "split",
    "question",
    "context_clean",
    "citation_text",
    "answer_gold",
    "final_approved",
    "accuracy",
    "groundedness",
    "doctrine_style",
    "usefulness",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")


# ------------------------------------------------------------
# 4. final_approved 처리
# ------------------------------------------------------------
# CSV에 TRUE/FALSE, True/False, 1/0, yes/no가 섞여 있어도 처리
def to_bool(x):
    if pd.isna(x):
        return False
    x = str(x).strip().lower()
    return x in ["true", "1", "yes", "y", "approved"]


df["final_approved_bool"] = df["final_approved"].apply(to_bool)


# ------------------------------------------------------------
# 5. 점수 숫자 변환
# ------------------------------------------------------------
score_cols = ["accuracy", "groundedness", "doctrine_style", "usefulness"]

for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)


# ------------------------------------------------------------
# 6. 학습 투입 조건
# ------------------------------------------------------------
# 처음 실험할 때 final_approved가 전부 FALSE이면 학습 데이터가 0개가 된다.
# 이 경우 두 가지 방식 중 하나를 선택해야 한다.
#
# A안: 네가 직접 승인한 final_approved=TRUE만 사용
# B안: 임시 실험용으로 점수 조건만 만족하면 사용
#
# 실제 최종 학습은 A안을 권장한다.
# 단, 최초 테스트에서는 B안을 사용할 수 있다.

USE_ONLY_FINAL_APPROVED = False  # 최종 학습 시 True 권장

if USE_ONLY_FINAL_APPROVED:
    trainable = df[
        (df["final_approved_bool"] == True)
        & (df["accuracy"] >= 4)
        & (df["groundedness"] >= 4)
        & (df["doctrine_style"] >= 4)
        & (df["usefulness"] >= 3)
        & (df["question"].notna())
        & (df["context_clean"].notna())
        & (df["answer_gold"].notna())
    ].copy()
else:
    trainable = df[
        (df["accuracy"] >= 4)
        & (df["groundedness"] >= 4)
        & (df["doctrine_style"] >= 4)
        & (df["usefulness"] >= 3)
        & (df["question"].notna())
        & (df["context_clean"].notna())
        & (df["answer_gold"].notna())
    ].copy()

print("학습 사용 가능 행 수:", len(trainable))

if len(trainable) == 0:
    raise ValueError("학습 가능한 행이 없습니다. final_approved 또는 점수 조건을 확인하세요.")


# ------------------------------------------------------------
# 7. 시스템 프롬프트
# ------------------------------------------------------------
SYSTEM_PROMPT = """너는 미 육군 교리 기반 RAG 답변 보조자다.
반드시 제공된 검색 근거 범위 안에서만 답변한다.
답변은 다음 5개 항목을 모두 포함해야 한다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항

답변 본문에는 chunk_id, chunk, 청크 같은 내부 데이터 관리 용어를 절대 사용하지 않는다.
교범명, 장·절, 페이지, 핵심 근거만 사용자가 이해할 수 있는 방식으로 제시한다.
근거가 부족하면 단정하지 말고, 검색된 자료 범위와 추가 확인 필요성을 명시한다.
실제 작전 실행명령, 특정 표적 타격 우선순위, 민감한 세부 전술은 일반 교리 원칙과 참모 검토사항 수준으로 제한한다."""


# ------------------------------------------------------------
# 8. JSONL 레코드 생성 함수
# ------------------------------------------------------------
def make_user_prompt(row):
    question = str(row["question"]).strip()
    context_clean = str(row["context_clean"]).strip()
    citation_text = str(row["citation_text"]).strip()

    user_prompt = f"""[검색 근거]
{context_clean}

[출처 정보]
{citation_text}

[질문]
{question}

위 검색 근거와 출처 정보만 바탕으로 답변하라.
답변은 반드시 다음 형식을 따른다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항"""

    return user_prompt


def make_jsonl_record(row):
    answer = str(row["answer_gold"]).strip()

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": make_user_prompt(row),
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ]
    }


# ------------------------------------------------------------
# 9. split별 저장
# ------------------------------------------------------------
def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in rows.iterrows():
            record = make_jsonl_record(row)
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


train_df = trainable[trainable["split"].astype(str).str.lower() == "train"].copy()
val_df = trainable[trainable["split"].astype(str).str.lower().isin(["validation", "val"])].copy()
test_df = trainable[trainable["split"].astype(str).str.lower() == "test"].copy()

# split이 비어 있거나 불균형하면 자동 분할
if len(train_df) == 0:
    train_df = trainable.sample(frac=0.8, random_state=42)
    remain_df = trainable.drop(train_df.index)
    val_df = remain_df.sample(frac=0.5, random_state=42)
    test_df = remain_df.drop(val_df.index)

write_jsonl(train_df, TRAIN_JSONL)
write_jsonl(val_df, VAL_JSONL)
write_jsonl(test_df, TEST_JSONL)

print("저장 완료")
print("train:", len(train_df), TRAIN_JSONL)
print("validation:", len(val_df), VAL_JSONL)
print("test:", len(test_df), TEST_JSONL)


# ------------------------------------------------------------
# 10. 샘플 확인
# ------------------------------------------------------------
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])